In [2]:
from ddgs import DDGS
import pandas as pd
from urllib.parse import urlparse

data = pd.read_csv("businesses.csv")

rest_names = data["name"].dropna().astype(str).str.strip().tolist()
rest_names = [name for name in rest_names if name]

DIR_DOMAINS = {
    "google.com",
    "facebook.com",
    "instagram.com",
    "zomato.com",
    "swiggy.com",
    "justdial.com",
    "tripadvisor.com",
    "magicpin.in",
    "eazydiner.com",
    "district.in",
    "reddit.com",
    "wikipedia.org",
    "indiashopping.io",
    "indorerocks.com",
    "treebo.com",
}

def clean_domain(url):
    host = urlparse(url).netloc.lower()
    if host.startswith("www."):
        host = host[4:]
    return host

def classify_link(url):
    domain = clean_domain(url)
    if domain in DIR_DOMAINS:
        return "relevant"
    return "official_or_other"

rows = []

with DDGS() as ddgs:
    for name in rest_names:
        query = f"{name} Indore"
        search_results = list(ddgs.text(query, max_results=5))

        seen = set()
        official_link = None
        relevant_links = []

        for r in search_results:
            url = r.get("href") or r.get("url")
            title = (r.get("title") or "").strip()
            body = (r.get("body") or "").strip()
            if not url or url in seen:
                continue
            seen.add(url)

            domain = clean_domain(url)
            link_row = {
                "name": name,
                "query": query,
                "title": title,
                "url": url,
                "domain": domain,
                "snippet": body,
                "type": classify_link(url),
            }

            if official_link is None and domain not in DIR_DOMAINS:
                official_link = link_row
                continue

            if len(relevant_links) < 4:
                relevant_links.append(link_row)

        rows.append({
            "name": name,
            "query": query,
            "has_website": official_link is not None,
            "official_url": official_link["url"] if official_link else "",
            "official_title": official_link["title"] if official_link else "",
            "relevant_links": relevant_links,
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("restaurant_online_presence.csv", index=False)
results_df.head()

,name,query,has_website,official_url,official_title,relevant_links
0,Indigo,Indigo Indore,True,https://en.wikipedia.org/wiki/Indigo_Index,Indigo Index,"[{'name': 'Indigo', 'query': 'Indigo Indore', ..."
1,Jain Mithai Bhandar(JMB),Jain Mithai Bhandar(JMB) Indore,True,https://restaurant-guru.in/JMB-Indore-4,"JMB - Jain Mithai Bhandar, Moti Tabela, Indore...","[{'name': 'Jain Mithai Bhandar(JMB)', 'query':..."
2,Apna Sweets,Apna Sweets Indore,True,https://apnasweets.com/,Apna Sweets |Order sweets online |Buy online s...,"[{'name': 'Apna Sweets', 'query': 'Apna Sweets..."
3,Bhojanalay and Tiffin Centre,Bhojanalay and Tiffin Centre Indore,True,https://yappe.in/madhya-pradesh/indore/bhojana...,Bhojanam the bhojnalaya & indore tiffin centre...,"[{'name': 'Bhojanalay and Tiffin Centre', 'que..."
4,MY Kitchen Restaurant,MY Kitchen Restaurant Indore,True,https://www.tripadvisor.in/Restaurant_Review-g...,INDORE KITCHEN RESTAURANT - Restaurant... - Tr...,"[{'name': 'MY Kitchen Restaurant', 'query': 'M..."


In [ ]:
from ddgs import DDGS

query = "Jain Mithai Bhandar(JMB) Indore"
with DDGS() as ddgs:
    results = list(ddgs.text(query, max_results=8))

for r in results[:5]:
    print(r.get("title"), "->", r.get("href") or r.get("url"))

[{'title': 'Jain Mithai Bhandar | Indore - Facebook', 'href': 'https://www.facebook.com/jainmithaibhandar/', 'body': 'Jain Mithai Bhandar (JMB) is Synonymous to Mithaiyaan & Festivals in Indore. - 2/2, Moti Tabela, Op'}, {'title': 'JMB - Jain Mithai Bhandar, HIG-LIG, Indore - Zomato', 'href': 'https://www.zomato.com/indore/jmb-jain-mithai-bhandar-hig-lig', 'body': 'JMB - Jain Mithai Bhandar, Mithai, North Indian, South Indian, Chinese, Sandwich, Fast Food, Street Food, Beverages, C/2, Sector 2, Ravi Shankar Shukla Nagar, ...'}, {'title': 'Jain Mithai Bhandar in Moti Tabela,Indore - Order Food Online', 'href': 'https://www.justdial.com/Indore/Jain-Mithai-Bhandar-Opposite-Collectorate-Office-Moti-Tabela/0731P731STD300017_BZDET', 'body': 'Many customers mentioned the wide variety of sweets and snacks available at Jain Mithai Bhandar (JMB).Some reviewers appreciated the taste and quality of ...'}, {'title': 'Order Food Online from Jain Mithai Bhandar (JMB) in Khajrana, Indore', 'href': 'ht

Dataset preprocessing

In [4]:
import ast
import re
from urllib.parse import urlparse

import pandas as pd

data = pd.read_csv("restaurant_online_presence.csv")

GENERIC_NAME_WORDS = {
    "and", "the", "a", "an", "of", "in", "at", "by", "for",
    "restaurant", "restaurants", "restro", "cafe", "cafeteria", "kitchen",
    "hotel", "hotels", "dhaba", "bhojanalay", "bhojnalaya", "tiffin",
    "centre", "center", "food", "foods", "sweets", "sweet", "bakery",
    "indore", "mp", "madhya", "pradesh",
}

KNOWN_UNOFFICIAL_DOMAINS = {
    "google", "facebook", "instagram", "zomato", "swiggy", "justdial",
    "tripadvisor", "restaurant-guru", "yappe", "wikipedia", "latlong",
    "youtube", "oyorooms", "agoda", "trip", "wanderlog", "nearbuy",
    "magicpin", "eazydiner", "menuweb", "yummraj", "mappls", "waze",
    "pincode", "yandex", "mapsofindia", "tradeindia", "allevents",
    "mandap", "franchisebyte", "prioritypass", "adanione", "hospemag",
    "timesofindia", "hindustantimes", "clearcarrental", "hotelsearchsite",
    "indiashopping", "viplounges", "saifeehoteltulear", "codepin",
}

INDIAN_2_PART_SUFFIXES = {"co.in", "com.in", "net.in", "org.in", "firm.in", "gen.in", "ind.in"}

def compact(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

def clean_host(url):
    host = urlparse(str(url)).netloc.lower()
    for prefix in ("www.", "m.", "amp."):
        if host.startswith(prefix):
            host = host[len(prefix):]
    return host

def domain_core(url):
    parts = clean_host(url).split(".")
    if len(parts) >= 3 and ".".join(parts[-2:]) in INDIAN_2_PART_SUFFIXES:
        return parts[-3]
    return parts[-2] if len(parts) >= 2 else ""

def important_name_tokens(name):
    raw_tokens = re.findall(r"[a-z0-9]+", str(name).lower())
    tokens = [t for t in raw_tokens if len(t) > 1 and t not in GENERIC_NAME_WORDS]
    if len(tokens) < 2:
        location_words = {"indore", "mp", "madhya", "pradesh"}
        tokens = [t for t in raw_tokens if len(t) > 1 and t not in location_words]
    return tokens

def is_official_link(name, url):
    if pd.isna(url) or not str(url).strip():
        return False, "missing url"

    core = domain_core(url)
    core_clean = compact(core)
    if not core_clean:
        return False, "bad url"
    if core_clean in {compact(d) for d in KNOWN_UNOFFICIAL_DOMAINS}:
        return False, "listing/referral domain"

    tokens = important_name_tokens(name)
    if not tokens:
        return False, "not enough restaurant-name words"

    name_clean = compact("".join(tokens))
    abbreviations = [compact(a) for a in re.findall(r"\(([a-zA-Z0-9]{2,6})\)", str(name))]
    token_hits = [t for t in tokens if len(t) >= 3 and t in core_clean]

    if len(name_clean) >= 4 and (name_clean in core_clean or core_clean in name_clean):
        return True, "domain matches restaurant name"
    if len(token_hits) >= min(2, len(tokens)):
        return True, "domain contains restaurant-name words"
    if any(len(a) >= 2 and (core_clean.startswith(a) or core_clean.endswith(a)) for a in abbreviations):
        return True, "domain contains restaurant abbreviation"
    return False, "domain does not match restaurant name"

def candidate_links(row):
    links = []
    if str(row.get("official_url", "")).strip():
        links.append(row["official_url"])

    try:
        relevant_links = ast.literal_eval(row.get("relevant_links", "[]"))
    except (ValueError, SyntaxError):
        relevant_links = []

    for item in relevant_links:
        if isinstance(item, dict) and str(item.get("url", "")).strip():
            links.append(item["url"])

    return list(dict.fromkeys(links))

def find_official_url(row):
    last_reason = "no candidate links"
    for url in candidate_links(row):
        ok, reason = is_official_link(row["name"], url)
        if ok:
            return url, True, reason
        last_reason = reason
    return "", False, last_reason

checked = data.apply(find_official_url, axis=1, result_type="expand")
data["original_official_url"] = data["official_url"]
data[["official_url", "has_official_website", "official_check_reason"]] = checked
data["has_website"] = data["has_official_website"]
data["needs_manual_review"] = data["original_official_url"].astype(str).str.strip().ne("") & ~data["has_official_website"]

data.to_csv("restaurant_online_presence_official_only.csv", index=False)

data[["name", "has_official_website", "official_url", "official_check_reason", "needs_manual_review"]].head(20)


,name,has_official_website,official_url,official_check_reason,needs_manual_review
0,Indigo,False,,domain does not match restaurant name,True
1,Jain Mithai Bhandar(JMB),False,,listing/referral domain,True
2,Apna Sweets,True,https://apnasweets.com/,domain matches restaurant name,False
3,Bhojanalay and Tiffin Centre,False,,listing/referral domain,True
4,MY Kitchen Restaurant,False,,domain does not match restaurant name,True
5,hazi samosa center,False,,listing/referral domain,True
6,Mr. Singh's Restaurant,False,,listing/referral domain,True
7,Nafees,False,,listing/referral domain,True
8,Saifee Restaurant,False,,domain does not match restaurant name,True
9,Saifee Hotel,False,,listing/referral domain,True


In [10]:
import pandas as pd
import ast

data = pd.read_csv("restaurant_online_presence_official_only.csv")

REFERRAL_DOMAINS = [
    "zomato.com",
    "swiggy.com",
    "justdial.com",
    "tripadvisor.com",
    "restaurant-guru.in",
    "eazydiner.com",
    "magicpin.in"
]

for domain in REFERRAL_DOMAINS:
    col = domain.replace(".", "_").replace("-", "_")
    data[col] = 0

data["total_referral_links"] = 0

for idx, row in data.iterrows():

    try:
        links = ast.literal_eval(row["relevant_links"])
    except:
        links = []

    referral_count = 0

    for link in links:

        domain = str(link.get("domain", "")).lower()

        if domain in REFERRAL_DOMAINS:

            col = domain.replace(".", "_").replace("-", "_")

            data.at[idx, col] += 1
            referral_count += 1

    data.at[idx, "total_referral_links"] = referral_count

data.to_csv("refferal count.csv")

In [11]:
import pandas as pd

data = pd.read_csv("refferal count.csv")

def calculate_score(row):

    score = 10 if row["has_official_website"] else 70

    unique_platforms = sum([
        row["zomato_com"] > 0,
        row["swiggy_com"] > 0,
        row["justdial_com"] > 0,
        row["tripadvisor_com"] > 0,
        row["restaurant_guru_in"] > 0,
        row["eazydiner_com"] > 0,
        row["magicpin_in"] > 0
    ])

    score += max(0, (3 - unique_platforms) * 5)

    if row["official_check_reason"] == "domain does not match restaurant name":
        score += 10

    return min(score, 100)

# Create lead score column
data["lead_score"] = data.apply(calculate_score, axis=1)
data["lead_score"].head()

0    70
1    15
2    75
3    80
4    85
Name: lead_score, dtype: int64

In [12]:
def lead_category(score):
    if score <= 20:
        return "Strong Online Presence"
    elif score <= 40:
        return "Low Priority"
    elif score <= 60:
        return "Medium Priority"
    elif score <= 80:
        return "High Priority"
    else:
        return "Hot Lead"

data["lead_category"] = data["lead_score"].apply(lead_category)

final_df = data[
    [
        "name",
        "official_url",
        "has_official_website",
        "lead_score",
        "lead_category"
    ]
]

final_df.to_csv("restaurant_leads_scored.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [ ]:
import selenium

